# 12 策略分析专题

覆盖 feature_trend_by_time / feature_drift_comparison / feature_effectiveness_by_segment /
feature_cross_heatmap / population_drift_monitor / segment_scorecard_comparison /
psi_cross_analysis / stability_report。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (germancredit, init_setting, OptimalBinning, WOEEncoder, LogisticRegression,
    feature_trend_by_time, feature_drift_comparison, feature_effectiveness_by_segment,
    feature_cross_heatmap, population_drift_monitor, segment_scorecard_comparison,
)
from hscredit.core.eda.stability import (
    psi_analysis, batch_psi_analysis, csi_analysis,
    time_psi_tracking, stability_report, psi_cross_analysis,
)
init_setting()
np.random.seed(42)
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
# 模拟时间列
start = pd.Timestamp('2022-01-01')
df['apply_date'] = pd.to_datetime([start + pd.Timedelta(days=int(d)) for d in np.random.randint(0,730,len(df))])
df['apply_month'] = df['apply_date'].dt.to_period('M').astype(str)
df['channel'] = np.random.choice(['线上','线下','中介'], len(df))
print(df[['apply_month','channel',target]].head(3))

## 1. feature_trend_by_time — 特征随时间趋势

In [ ]:
# 均值趋势
fig = feature_trend_by_time(df, 'credit.amount', 'apply_date', stat='mean', freq='Q')
plt.show()

In [ ]:
# 坏率趋势
fig = feature_trend_by_time(df, 'duration.in.month', 'apply_date',
    target=target, stat='badrate', freq='Q')
plt.show()

In [ ]:
# PSI趋势（相对第一期）
fig = feature_trend_by_time(df, 'credit.amount', 'apply_date', stat='psi', freq='Q')
plt.show()

## 2. feature_drift_comparison — 特征偏移对比

In [ ]:
df_tr, df_te = train_test_split(df, test_size=0.3, random_state=42)
fig = feature_drift_comparison(df_tr[num_feats], df_te[num_feats], num_feats, top_n=10)
plt.show()

## 3. feature_effectiveness_by_segment — 分客群特征有效性

In [ ]:
fig = feature_effectiveness_by_segment(df, 'credit.amount', target, 'channel', metric='iv')
plt.show()

In [ ]:
fig = feature_effectiveness_by_segment(df, 'duration.in.month', target, 'channel', metric='ks')
plt.show()

## 4. feature_cross_heatmap — 两特征交叉热力图

In [ ]:
# 坏率热力图
fig = feature_cross_heatmap(df, 'duration.in.month', 'credit.amount', target, stat='badrate', n_bins=5)
plt.show()

In [ ]:
# LIFT热力图
fig = feature_cross_heatmap(df, 'duration.in.month', 'credit.amount', target, stat='lift', n_bins=5)
plt.show()

## 5. population_drift_monitor — 多期客群偏移监控

In [ ]:
months = sorted(df['apply_month'].unique())[:4]
df_list = [df[df['apply_month']==m] for m in months]
fig = population_drift_monitor(df_list, labels=months, features=num_feats[:6], top_n_drift=3)
plt.show()

## 6. segment_scorecard_comparison — 分客群评分效果

In [ ]:
woe = WOEEncoder(max_n_bins=5)
X_woe = woe.fit_transform(df[num_feats], df[target])
lr = LogisticRegression(max_iter=1000); lr.fit(X_woe, df[target])
df['score'] = lr.predict_proba(X_woe)[:,1]
fig = segment_scorecard_comparison(df, 'score', target, 'channel', metrics=['KS','AUC'])
plt.show()

## 7. psi_cross_analysis — PSI交叉矩阵

In [ ]:
# 按时间自动分组
result = psi_cross_analysis(df, ['credit.amount','duration.in.month'],
    date_col='apply_date', freq='Q', return_matrix=True)
print('credit.amount PSI矩阵:')
print(result['credit.amount'].round(4))

In [ ]:
# 按渠道手工分组
result2 = psi_cross_analysis(df, ['credit.amount'], group_col='channel', return_matrix=False)
result2

## 8. time_psi_tracking — PSI时序追踪

In [ ]:
tracking = time_psi_tracking(df, num_feats[:4], 'apply_date', freq='Q')
print(tracking)
# 透视为矩阵
if not tracking.empty:
    print('\nPSI矩阵:')
    print(tracking.pivot_table(index='时间周期', columns='特征名', values='PSI值').round(4))

## 9. stability_report — 综合稳定性报告

In [ ]:
rep = stability_report(df, num_feats[:5], 'apply_date', psi_threshold=0.1)
rep